# 07 — RL agent (walk 1, PPO + cost-variant sweep)

PPO policy that allocates weights over the walk-1 ranker top-30 each Friday.
Trains separate policies at 4 transaction-cost levels (5/2/10/20 bps) so one
overnight run produces all the cost-sensitivity ablations for the paper.

**Spec:** `docs/superpowers/specs/2026-05-17-rl-agent-design.md`.

SB3 conventions: `check_env` validation → `EvalCallback` (best by val) →
`VecNormalize` (saved for backtest reuse) → TensorBoard. Single-walk MVP;
walks 2-16 deferred. Backtest = notebook 08.

## A. Setup

In [ ]:
from __future__ import annotations
import json, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, ProgressBarCallback,
)

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import friday_only, project_text_to_pca, load_walk_pca
from src.utils.rl_env import (
    PortfolioEnv,
    build_scoreboard_from_scored_panel,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'

PANEL_DIR  = processed_dir() / 'training_panel'
EMBED_DIR  = processed_dir() / 'finbert_stockday_embed'
RANKER_DIR = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_ROOT   = repo_root() / 'artifacts' / 'rl' / f'walk-{WALK_ID:03d}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Cost sweep — primary first; resumable across costs.
COSTS_BPS = [5, 2, 10, 20]
TOP_K = 30
EPISODE_LENGTH = 52
MAX_WEIGHT = 0.10
TOTAL_TIMESTEPS = 2_000_000
EVAL_FREQ = 10_000
CHECKPOINT_FREQ = 200_000
N_ENVS = 4
RANDOM_STATE = 42

print(f'walk {WALK_ID}; train {TRAIN_START}..{TRAIN_END}, val {VAL_START}..{VAL_END}')
print(f'cost sweep: {COSTS_BPS} bps; {TOTAL_TIMESTEPS:,} timesteps per variant')
print(f'out_root: {OUT_ROOT.relative_to(repo_root())}')